In [6]:
"""
共享 CNN 前端：所有模型使用完全相同的降采样模块
输入: (batch, 2, 1024) → 输出: (batch, 128, 128)
序列长度 8 倍压缩，特征维度 2→128
"""
import torch
import torch.nn as nn

In [8]:
class CNNFrontend(nn.Module):
    def __init__(self, in_channels=2, out_channels=128):
        super().__init__()
        self.cnn = nn.Sequential(
            # Block 1:(B, 2, 1024) -> (B, 64, 512)
            nn.Conv1d(in_channels, 64, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.1),

            # Block 2:(B, 64, 512) -> (B, 128, 256)
            nn.Conv1d(64, out_channels, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(out_channels),
            nn.GELU(),
            nn.Dropout(0.1),

            # Block 3:(B, 128, 256) -> (B, 128, 128)
            nn.Conv1d(out_channels, out_channels, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm1d(out_channels),
            nn.GELU(),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.cnn(x)
    
# 共享的分类头（所有模型都用同样的结构）
class ClassifierHead(nn.Module):
    def __init__(self, in_features, num_classes):
        super().__init__()
        self.head = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(in_features, 128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.head(x)
    
if __name__ == '__main__':
    cnn = CNNFrontend()
    x = torch.randn(4, 2, 2048)
    out = cnn(x)
    print(f"CNN Fronted: {x.shape} -> {out.shape}")
    print(f"Parameters:{sum(p.numel() for p in cnn.parameters()):,}")

CNN Fronted: torch.Size([4, 2, 2048]) -> torch.Size([4, 128, 256])
Parameters:91,968


# Conv1d 输出长度：公式与参数分类

## 一、核心公式

Conv1d 的输出长度公式（**PyTorch 自动计算**）：

$$L_{out} = \left\lfloor \frac{L_{in} + 2 \times padding - dilation \times (kernel\_size - 1) - 1}{stride} + 1 \right\rfloor$$

本模型中 `dilation=1`（默认值），简化为：

$$\boxed{L_{out} = \left\lfloor \frac{L_{in} + 2p - k}{s} + 1 \right\rfloor}$$

---

## 二、逐层追踪长度变化

### Block 1

```python
nn.Conv1d(in_channels=2, out_channels=64, kernel_size=7, stride=2, padding=3)
```

$$L_{out} = \left\lfloor \frac{1024 + 2 \times 3 - 7}{2} + 1 \right\rfloor = \left\lfloor \frac{1023}{2} + 1 \right\rfloor = \lfloor 512.5 \rfloor = 512$$

```
(B, 2, 1024) → (B, 64, 512)    长度减半
```

### Block 2

```python
nn.Conv1d(64, 128, kernel_size=5, stride=2, padding=2)
```

$$L_{out} = \left\lfloor \frac{512 + 2 \times 2 - 5}{2} + 1 \right\rfloor = \left\lfloor \frac{511}{2} + 1 \right\rfloor = 256$$

```
(B, 64, 512) → (B, 128, 256)   长度减半
```

### Block 3

```python
nn.Conv1d(128, 128, kernel_size=3, stride=2, padding=1)
```

$$L_{out} = \left\lfloor \frac{256 + 2 \times 1 - 3}{2} + 1 \right\rfloor = \left\lfloor \frac{255}{2} + 1 \right\rfloor = 128$$

```
(B, 128, 256) → (B, 128, 128)  长度减半
```

### 总览

```
(B, 2, 1024) → (B, 64, 512) → (B, 128, 256) → (B, 128, 128)
        ×1/2          ×1/2           ×1/2
```

> **每个 Block 长度减半，总共缩短 8 倍：1024 → 128**

---

## 三、参数分类

### ① 由数据决定（给定的）

```python
in_channels = 2       # 输入有几个通道（IQ两路信号）
L_in = 1024           # 输入序列长度（数据采集决定的）
B = batch_size        # 批大小（DataLoader 决定的）
```

### ② 由你设计（超参数，自己选的）

```python
out_channels = 64, 128   # 通道数：控制模型容量
kernel_size  = 7, 5, 3   # 卷积核大小：控制局部感受野
stride       = 2, 2, 2   # 步长：控制下采样倍率
dropout      = 0.1       # 正则化强度
```

### ③ 需要你手动计算的

```python
padding = ???   # 你必须自己算，PyTorch 不会帮你选
```

**设计意图是"stride=2 时长度恰好减半"**，倒推出 padding 的规律：

```python
# 当 stride=2，想让 L_out = L_in / 2 时：
padding = (kernel_size - 1) // 2
```

验证：
```
kernel_size=7 → padding = (7-1)//2 = 3  ✅
kernel_size=5 → padding = (5-1)//2 = 2  ✅
kernel_size=3 → padding = (3-1)//2 = 1  ✅
```

### ④ PyTorch 自动计算的

```python
L_out  # 输出序列长度 —— 你不需要写代码算，PyTorch 根据公式自动得出
       # 但你需要自己心里清楚（写注释、设计后续层时要用到）
```

---

## 四、对比总结表

```
                给定    你选的   你算的   PyTorch自动算的
─────────────────────────────────────────────────────
in_channels      ✅
L_in (1024)      ✅
out_channels             ✅
kernel_size              ✅
stride                   ✅
dropout                  ✅
padding                           ✅
L_out                                      ✅
─────────────────────────────────────────────────────
```

**一句话总结：**

> **你选 `kernel_size` 和 `stride`，然后根据"想要的下采样倍率"用公式算出 `padding`，而输出长度 `L_out` 是 PyTorch 根据这些参数自动计算出来的。**

# L_in 从 1024 改为 2048：怎么处理？

## 一、先看不改模型会发生什么

直接把 2048 喂进去，Conv1d **不会报错**，只是输出变长了：

```
原来 L_in=1024:
(B, 2, 1024) → (B, 64, 512) → (B, 128, 256) → (B, 128, 128)

现在 L_in=2048（不改模型）:
(B, 2, 2048) → (B, 64, 1024) → (B, 128, 512) → (B, 128, 256)
                                                         ↑
                                                  输出变成 256 而非 128
```

**Conv1d 本身对序列长度没有限制**，它是滑动窗口操作，任意长度都能跑。

---

## 二、会不会出问题？取决于后续模块

```
CNNFrontend 输出 (B, 128, L)
        │
        ▼
   后续是什么？
   ├── TCN Block      → ✅ 不会报错（Conv1d 不限长度）
   ├── Global Pooling  → ✅ 不会报错（池化到固定维度）
   └── Linear 全连接   → ❌ 会报错！（期望固定输入维度）
```

### 情况1：后续是 TCN + Global Pooling → **不用改，自适应**

```python
# TCN 不关心序列长度
# 最后 Global Average Pooling 会把任意长度压成 1
x = x.mean(dim=-1)  # (B, 128, 任意长度) → (B, 128)
```

> **这种情况下模型天然自适应，什么都不用改。**

### 情况2：后续有 Linear 层依赖固定长度 → **必须改**

---

## 三、如果必须保持输出 128，三种方案对比

### 方案 A：在末尾加一个 Block（最简单 ✅推荐）

```python
self.cnn = nn.Sequential(
    # Block 1: (B, 2, 2048) → (B, 64, 1024)
    nn.Conv1d(2, 64, kernel_size=7, stride=2, padding=3),
    nn.BatchNorm1d(64), nn.GELU(), nn.Dropout(0.1),

    # Block 2: (B, 64, 1024) → (B, 128, 512)
    nn.Conv1d(64, 128, kernel_size=5, stride=2, padding=2),
    nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(0.1),

    # Block 3: (B, 128, 512) → (B, 128, 256)
    nn.Conv1d(128, 128, kernel_size=3, stride=2, padding=1),
    nn.BatchNorm1d(128), nn.GELU(),

    # Block 4（新增）: (B, 128, 256) → (B, 128, 128)  ← 多降采样一次
    nn.Conv1d(128, 128, kernel_size=3, stride=2, padding=1),
    nn.BatchNorm1d(128), nn.GELU(),
)
```

### 方案 B：在开头加一个 Block

```python
self.cnn = nn.Sequential(
    # Block 0（新增）: (B, 2, 2048) → (B, 32, 1024)
    nn.Conv1d(2, 32, kernel_size=7, stride=2, padding=3),
    nn.BatchNorm1d(32), nn.GELU(), nn.Dropout(0.1),

    # Block 1: (B, 32, 1024) → (B, 64, 512)  ← in_channels 改成 32
    nn.Conv1d(32, 64, kernel_size=7, stride=2, padding=3),
    nn.BatchNorm1d(64), nn.GELU(), nn.Dropout(0.1),

    # Block 2: (B, 64, 512) → (B, 128, 256)
    nn.Conv1d(64, 128, kernel_size=5, stride=2, padding=2),
    nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(0.1),

    # Block 3: (B, 128, 256) → (B, 128, 128)
    nn.Conv1d(128, 128, kernel_size=3, stride=2, padding=1),
    nn.BatchNorm1d(128), nn.GELU(),
)
```

### 方案 C：不加层，改 stride（不推荐）

```python
# 把某一层的 stride 从 2 改成 4，一次降 4 倍
nn.Conv1d(2, 64, kernel_size=7, stride=4, padding=3)  # 2048 → 512
```

> 信息损失太大，不推荐。

---

## 四、三种方案对比

```
              改动量    参数增加    信息保留    推荐度
方案A 末尾加层   小       少        好         ⭐⭐⭐
方案B 开头加层   中       中        最好       ⭐⭐
方案C 改stride   小       无        差         ⭐
什么都不改       无       无        好         ⭐⭐⭐（如果后续有全局池化）
```

---

## 五、最终建议

```
                  ┌─ 后续有 Global Pooling ─→ 什么都不用改 ✅
                  │
L_in: 1024→2048 ─┤
                  │
                  └─ 后续有 Linear 需要固定长度 ─→ 末尾加一个 Block 4 ✅
                        输入翻倍 → 多降采样一次 → 输出长度不变
```

> **核心原则：输入长度翻倍 = 多一次 stride=2 降采样。加在末尾最简单，不影响已有层的设计。**